# Hugging Face Pipelines and Fine-Tuning
Let's finish the task of **Text classification** using IMDb sentiment analysis.

The goal is not only to run code, but also to understand the workflow:

- choose a pretrained model,
- load and preprocess a dataset,
- fine-tune the model,
- save the model,
- reload it with `pipeline()`,
- test the final system,
- discuss results and limitations.

## 0. Setup

Run the following cell first. In Google Colab, use **Runtime → Change runtime type → GPU** when possible.

The full datasets can be large, so this notebook uses **small subsets** for teaching purposes. Students may increase the subset size if they have enough compute.

In [16]:
#!pip install -q transformers datasets evaluate accelerate torch
#!pip install -U pip
#!pip install transformers==4.38.2 \
#    accelerate==0.27.2 \
#    tokenizers==0.15.2 \
#    datasets==2.18.0 \
#    evaluate==0.4.1 \
#    sentencepiece \
#    torch \
#    ipykernel
!pip install "accelerate==0.27.2"

/bin/bash: line 1: /home/user/ai-course/.venv/bin/pip: cannot execute: required file not found


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# Text Classification with IMDb
- Demonstrate basic usage of a sentiment-analysis pipeline.
- Fine-tune a pretrained text classification model using IMDb.
- Save the fine-tuned model.
- Reload the saved model with `pipeline()`.
- Classify sample texts and print results.

Let's try to use 
1. `pipeline()` for simple inference, 
2. `Trainer` for fine-tuning, 
3. `AutoTokenizer` for tokenization, and 
4. `AutoModelForSequenceClassification` classes for loading pretrained models.

We will use **DistilBERT** because it is smaller and faster than BERT, while still being a good Transformer model for teaching.

In [2]:
import numpy as np
import torch

import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, #convert text into model inputs.
    DataCollatorWithPadding, #dynamically pad inputs to the longest sequence in a batch.
    AutoModelForSequenceClassification, #load a pretrained model for text classification.
    TrainingArguments, #define training settings.
    Trainer,#train or fine-tune the model.
    pipeline,
)

checkpoint = "distilbert-base-uncased"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2026-06-09 00:25:37.091764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-09 00:25:37.129567: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-09 00:25:37.129670: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-09 00:25:37.189016: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-09 00:25:38.078867: W tensorflow/compiler/tf

Torch: 2.12.0+cu130
CUDA available: True


## 0 Baseline: Basic sentiment pipeline before fine-tuning

This demonstrates the simplest Hugging Face workflow. 

The `pipeline()` function hides many steps: loading a model, loading a tokenizer, tokenizing the input, running inference, and converting logits into labels and scores.

In [3]:
basic_classifier = pipeline("sentiment-analysis")

sample_texts = [
    "This movie was surprisingly good and very emotional.",
    "The plot was boring and the acting was terrible.",
    "It was okay, but I probably would not watch it again."
]

print("=== Basic pretrained sentiment pipeline ===")
for text in sample_texts:
    print(text, "=>", basic_classifier(text))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/home/user/ai-course/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


=== Basic pretrained sentiment pipeline ===
This movie was surprisingly good and very emotional. => [{'label': 'POSITIVE', 'score': 0.9998784065246582}]
The plot was boring and the acting was terrible. => [{'label': 'NEGATIVE', 'score': 0.9997890591621399}]
It was okay, but I probably would not watch it again. => [{'label': 'NEGATIVE', 'score': 0.9976522326469421}]


## 1 Load IMDb dataset

IMDb is a binary sentiment classification dataset:

- label `0` = negative,
- label `1` = positive.

To keep the notebook practical for class, we use only a small subset.

In [4]:
dataset = load_dataset("imdb")
dataset.shape
#Small subsets
train_ds = dataset["train"].shuffle(seed=42).select(range(2000))
test_ds = dataset["test"].shuffle(seed=42).select(range(500))

#train_ds = dataset["train"] 
#test_ds = dataset["test"] 

#print(train_ds)
#print(test_ds)
print(train_ds[0])
print(test_ds[0])

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}
{'text': "<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, 

## 2 Tokenization

Neural networks cannot directly process raw text. The tokenizer converts text into numerical token IDs.

1. For Transformer models, common tokenizer outputs include:
- `input_ids`: integer IDs for tokens,
- `attention_mask`: tells the model which tokens are real and which are padding.

2. We use `truncation=True` to cut very long reviews, `padding="max_length"` to make all examples the same length, and `max_length=256` to reduce memory use.

3. `batched=True` : tells Hugging Face to tokenize many examples at the same time, which is faster than inputs["texts"]
- withput `batch=True`
{   "text": "I love this movie.",
    "label": 1}
- With `batch=True`
{
    "text": [
        "I love this movie.",
        "I hate this movie.",
        "This film is okay."
    ],
    "label": [1, 0, 1]
}

In [5]:
# Load the tokenizer for the pretrained model
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/home/user/ai-course/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
def tokenize_function(inputs):
    return tokenizer(
        inputs["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

# Remove raw text to save memory and match model input format.
# train_ds = train_ds.remove_columns(["text"])
# test_ds = test_ds.remove_columns(["text"])

# Hugging Face Trainer expects the target column to be named "labels".
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

# Not always necessary but recommand: return pytorch tensors instead of lists for model training.
#train_ds.set_format("torch")
#test_ds.set_format("torch")

print(train_ds[0])
print(train_ds[0].keys())



Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'labels': 1, 'input_ids': [101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337, 2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610, 2186, 2055, 6355, 6997, 1012, 6337, 2099, 3504, 15594, 2100, 1010, 3481, 3771, 35

## 3 Load model and define metric
`checkpoint = "distilbert-base-uncased"`

The base DistilBERT model is used to represent and understand the sentence, but it is not directly designed for IMDb sentiment classification.

When we use `AutoModel` , as we did earlier, the model only returns hidden states — contextual vector representations of the input tokens. It does not output POSITIVE or NEGATIVE labels.

Therefore, for sentiment classification, we use `AutoModelForSequenceClassification`. This loads a Transformer model with an additional classification head. The base DistilBERT model produces contextual text representations, and the classification head maps those representations to two sentiment labels: NEGATIVE and POSITIVE.

In [7]:
# This is a 2-class classification model.
# Class 0 means NEGATIVE. Class 1 means POSITIVE.
# mapping labels to human-readable form correctly
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

#pip install scikit-learn
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4. Fine-tune with `Trainer`

`TrainingArguments` defines the training settings.

`Trainer` runs the training loop for us.

For classroom demo, we use small settings:
- 1 epoch
- small batch size
- evaluation at the end of each epoch
- no external logging tools



To Fine-tune the model using our specific data, we need to design some important hyperparameters:

| Hyperparameter | Meaning |
|---|---|
| `learning_rate=2e-5` | Common small learning rate for Transformer fine-tuning |
| `batch_size=8` | Small enough for a demo |
| `num_train_epochs=1` | Fast classroom demo; students may increase |
| `weight_decay=0.01` | Regularization |


In [13]:
import transformers
import accelerate

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.38.2
accelerate: 1.13.0


In [12]:
training_args = TrainingArguments(
    output_dir="./imdb_training_checkpoints",
    #eval_strategy ="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

TypeError: Accelerator.__init__() got an unexpected keyword argument 'dispatch_batches'

### Start Training

In [11]:
trainer.train()

NameError: name 'trainer' is not defined

### Evaluate the Fine-tuned Model

In [ ]:
metrics = trainer.evaluate()
print(metrics)

## 5 Save and reload the fine-tuned model through `pipeline()`

This is important because the assignment asks students to use the trained model with the Hugging Face `pipeline()` function.

In [ ]:
OUTPUT_DIR = "./saved_imdb_sentiment_model"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
)

print("=== Fine-tuned IMDb sentiment pipeline ===")
for text in sample_texts:
    print(text, "=>", fine_tuned_classifier(text))

## 6 Discussion points for the report

Students should discuss:

- What pretrained model was selected and why?
- What dataset was used?
- How many examples were used for training and validation?
- What hyperparameters were selected?
- How did the fine-tuned model perform?
- Did the fine-tuned IMDb model behave differently from the default sentiment pipeline?
- What are the limitations?

Important limitation: IMDb is movie-review data. A model fine-tuned on IMDb may not generalize perfectly to product reviews, political comments, or social-media text.

In [ ]:
from transformers import (
    AutoModelForQuestionAnswering,
    default_data_collator,
)

QA_MODEL_NAME = "distilbert-base-uncased-distilled-squad"
QA_OUTPUT_DIR = "./saved_squad_qa_model"

In [ ]:
basic_qa = pipeline("question-answering", model=QA_MODEL_NAME)

context = """
Transformers are neural network architectures based on self-attention.
They are widely used in natural language processing tasks such as translation,
question answering, summarization, and text generation.
"""

question = "What mechanism are Transformers based on?"

print("=== Basic QA pipeline ===")
print(basic_qa(question=question, context=context))

In [ ]:
squad = load_dataset("squad")

# Small subsets for classroom practicality.
train_squad = squad["train"].shuffle(seed=42).select(range(2000))
valid_squad = squad["validation"].shuffle(seed=42).select(range(500))

print(train_squad)
print(train_squad[0])

In [ ]:
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)

max_length = 384
doc_stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = qa_tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end token indices of the context.
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside this context window, use CLS token position.
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Find start token.
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            # Find end token.
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

train_qa_dataset = train_squad.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=train_squad.column_names,
)

valid_qa_dataset = valid_squad.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=valid_squad.column_names,
)

print(train_qa_dataset)

In [ ]:
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)

qa_training_args = TrainingArguments(
    output_dir="./squad_training_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none",
)

qa_trainer = Trainer(
    model=qa_model,
    args=qa_training_args,
    train_dataset=train_qa_dataset,
    eval_dataset=valid_qa_dataset,
    tokenizer=qa_tokenizer,
    data_collator=default_data_collator,
)

qa_trainer.train()

In [ ]:
qa_trainer.save_model(QA_OUTPUT_DIR)
qa_tokenizer.save_pretrained(QA_OUTPUT_DIR)

fine_tuned_qa = pipeline(
    "question-answering",
    model=QA_OUTPUT_DIR,
    tokenizer=QA_OUTPUT_DIR,
)

qa_examples = [
    {
        "context": context,
        "question": "What tasks are Transformers used for?"
    },
    {
        "context": context,
        "question": "What mechanism are Transformers based on?"
    }
]

print("=== Fine-tuned SQuAD QA pipeline ===")
for item in qa_examples:
    print("Question:", item["question"])
    print("Answer:", fine_tuned_qa(question=item["question"], context=item["context"]))
    print()

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
)

GEN_MODEL_NAME = "distilgpt2"
GEN_OUTPUT_DIR = "./saved_story_generation_model"

In [ ]:
basic_generator = pipeline("text-generation", model=GEN_MODEL_NAME)

prompt = "In the future, artificial intelligence will"

print("=== Basic text generation pipeline ===")
print(
    basic_generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
    )[0]["generated_text"]
)

In [ ]:
texts = [
    "In a small village, a young engineer built a robot that could help farmers water their crops.",
    "The robot learned from the weather, the soil, and the sunlight. Every day, it made better decisions.",
    "One morning, the village faced a drought. The robot suggested a new irrigation plan.",
    "At first, the farmers were unsure. But after one week, the crops began to recover.",
    "The young engineer realized that technology works best when it supports people and nature.",
    "In another city, students used artificial intelligence to summarize long books and ask better questions.",
    "Their teacher reminded them that AI should be used as a learning assistant, not as a replacement for thinking.",
    "The students compared the AI answers with reliable sources and improved their final reports.",
]

story_dataset = Dataset.from_dict({"text": texts})
print(story_dataset)
print(story_dataset[0])

In [ ]:
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_tokenizer.pad_token = gen_tokenizer.eos_token

def tokenize_generation_examples(batch):
    return gen_tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_story_dataset = story_dataset.map(
    tokenize_generation_examples,
    batched=True,
    remove_columns=["text"],
)

print(tokenized_story_dataset[0].keys())

In [ ]:
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME)
gen_model.config.pad_token_id = gen_tokenizer.eos_token_id

data_collator = DataCollatorForLanguageModeling(
    tokenizer=gen_tokenizer,
    mlm=False,
)

gen_training_args = TrainingArguments(
    output_dir="./gpt2_training_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
)

gen_trainer = Trainer(
    model=gen_model,
    args=gen_training_args,
    train_dataset=tokenized_story_dataset,
    tokenizer=gen_tokenizer,
    data_collator=data_collator,
)

gen_trainer.train()

In [ ]:
gen_trainer.save_model(GEN_OUTPUT_DIR)
gen_tokenizer.save_pretrained(GEN_OUTPUT_DIR)

fine_tuned_generator = pipeline(
    "text-generation",
    model=GEN_OUTPUT_DIR,
    tokenizer=GEN_OUTPUT_DIR,
)

prompts = [
    "The young engineer built",
    "Artificial intelligence should help students",
]

print("=== Fine-tuned text generation pipeline ===")
for p in prompts:
    result = fine_tuned_generator(
        p,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        repetition_penalty=1.2,
    )
    print("
Prompt:", p)
    print(result[0]["generated_text"])